# AstroOS Python SDK — Quickstart

Compute a birth chart, run Dasha analysis, and generate a report against a **local** AstroOS instance.

Prerequisites: AstroOS API running at `http://localhost:8000` (see `README.md`), and `pip install astroos`.

In [ ]:
from astroos import AstroOSClient, SdkConfig, AstroOSError

client = AstroOSClient(base_url="http://localhost:8000/api/v1")

# Register / login (local instance)
EMAIL, PASSWORD = "notebook@example.com", "notebook-password-1"
try:
    client.auth.register(email=EMAIL, password=PASSWORD, display_name="Notebook User")
except AstroOSError:
    pass  # already registered
client.auth.login(email=EMAIL, password=PASSWORD)

## 1. Compute a D1 (Rasi) chart

In [ ]:
BIRTH = dict(
    birth_datetime_utc="1986-06-15T10:30:00+00:00",
    latitude=28.6139,   # New Delhi
    longitude=77.2090,
)

chart = client.chart.compute(**BIRTH, ayanamsa="lahiri", house_system="W")
for planet in chart["planets"][:3]:
    print(planet)

## 2. Vimshottari Dasha

In [ ]:
dasha = client.dasha.compute("vimshottari", **BIRTH, max_depth=2)
dasha["periods"][:2] if "periods" in dasha else dasha

## 3. Generate a report (and export PDF)

In [ ]:
report = client.reports.generate_chart(**BIRTH, title="Notebook Chart", subject_name="Notebook User")
pdf_bytes = client.reports.generate_pdf(**BIRTH)
with open("notebook_chart.pdf", "wb") as f:
    f.write(pdf_bytes)
print(f"PDF written: notebook_chart.pdf ({len(pdf_bytes)} bytes)")

## 4. Error handling

All SDK errors derive from `AstroOSError` (auth, validation, rate-limit, not-found, server).

In [ ]:
try:
    client.chart.compute(birth_datetime_utc="not-a-date", latitude=0.0, longitude=0.0)
except AstroOSError as exc:
    print(type(exc).__name__, "→", exc)